This notebook demonstrates how to run the forest deforestation User Defined Process

In [ ]:
import logging

from utils import urls

import openeo

logging.basicConfig(level=logging.INFO)

In [ ]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")

In [ ]:
connection.authenticate_oidc()

In [ ]:
spatial_extent = {
    "west": 30.55,
    "south": 1.07,
    "east": 31.23,
    "north": 1.55,
}

resample_spatial_resolution = 30  # m

# Load decimal year of deforestation

In [ ]:
# load results from previous batch job
JOB_ID = "j-26091510470147c4b69c962dacd24c7a"

deforestation_year = connection.load_stac_from_job(
    JOB_ID,
    spatial_extent=spatial_extent,
)

In [ ]:
deforestation_year = deforestation_year.drop_dimension("t")

# Run KPIs UDP

In [ ]:
kpis_vector_cube = connection.datacube_from_process(
    "KPIs",
    namespace=urls.KPIS_UDP,
    decimal_year_of_deforestation_datacube=deforestation_year,
    spatial_extent=spatial_extent,
)

In [ ]:
job = kpis_vector_cube.create_job(out_format="Parquet")

In [ ]:
job.start_and_wait()

In [ ]:
results = job.get_results()

In [ ]:
!mkdir -p output-udp/
!rm -r output-udp/

In [ ]:
results.download_files("output-udp/")

In [ ]:
import json

with open("logs.json", "w") as f:
    json.dump(job.logs(), f, indent=2)